# 01 - Baseline model and refusal detector (Section 1)
Runtime: **T4 GPU**. Requires `00_setup_and_splits` to have been run once.

In [ ]:
# 1) Mount Drive: data/, vectors/, outputs/, results/ are stored there and survive disconnects
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["STEERING_DATA_ROOT"] = "/content/drive/MyDrive/refusal-steering"
os.makedirs(os.environ["STEERING_DATA_ROOT"], exist_ok=True)

# 2) Get the code (CHANGE the URL to your repo)
REPO_URL = "https://github.com/<your-user>/refusal-steering.git"
if not os.path.exists("/content/refusal-steering"):
    !git clone {REPO_URL} /content/refusal-steering
else:
    !git -C /content/refusal-steering pull
%cd /content/refusal-steering

# 3) Install pinned dependencies (torch comes preinstalled with Colab)
!pip install -q -r requirements.txt

## 1. Smoke test
Loads Qwen3-4B in fp16 and generates for 3 prompts. Outputs must be coherent text (not empty or repeated characters).

In [ ]:
!python scripts/01_baseline.py --smoke

## 2. Baseline on the validation harmful prompts (Section 1.3)
Resumable: if the session disconnects, run this cell again and it continues where it stopped.

In [ ]:
!python scripts/01_baseline.py --split val --kind harmful

## 3. Review the refusal detector (Section 1.2)
Writes a CSV with 20 random outputs. Open it in Drive, fill `manual_label`, and compare with `detector_label`. If the detector is wrong, edit `PHRASES` in `src/refusal.py`, push, and re-run this cell and the previous one (no regeneration needed: outputs are cached).

In [ ]:
!python scripts/01b_review_refusals.py --split val --kind harmful --n 20

## 4. (Optional) Baseline on benign validation prompts
Quick check that the baseline does not refuse harmless requests.

In [ ]:
!python scripts/01_baseline.py --split val --kind harmless

## Gate before Section 2
- [ ] Smoke test outputs look normal
- [ ] Baseline refusal rate on val harmful >= 70%
- [ ] Detector agrees with manual labels on most of the 20 reviewed outputs

The test split stays locked until Section 4.